In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
# import psutil
# import logging

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

boco_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/boco_wind_vec.nc"
clust_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/cluster_wind_vec.nc"

output_dir = '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/R2_hourly_composites'

In [2]:
extent=[140, 155, -42, -28]

lon_min, lon_max, lat_min, lat_max = extent

In [3]:
client = Client(n_workers=12,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45503 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/45503/status,
Dashboard: /proxy/45503/status,Workers: 12
Total threads: 12,Total memory: 55.88 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38303,Workers: 0
Dashboard: /proxy/45503/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39001,Total threads: 1
Dashboard: /proxy/45961/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:37905,


2025-08-28 13:29:16,969 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b1a4d3161d71aeda92965133e1ffb379 initialized by task ('open_dataset-rechunk-transfer-9a996f5adbf6a9bb8e39cf31291f1313', 0, 0, 0, 99, 0, 0) executed on worker tcp://127.0.0.1:34361
2025-08-28 13:29:24,466 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 6cf5db1f9543ab45270cd01db7a588f8 initialized by task ('open_dataset-rechunk-transfer-85797b91985c33e0bea9a36b76768669', 0, 0, 0, 99, 0, 0) executed on worker tcp://127.0.0.1:34633
2025-08-28 13:29:50,738 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b1a4d3161d71aeda92965133e1ffb379 deactivated due to stimulus 'task-finished-1756351790.7372835'
2025-08-28 13:30:13,844 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 6cf5db1f9543ab45270cd01db7a588f8 deactivated due to stimulus 'task-finished-1756351813.842351'


In [4]:
# def setup_dask_client(
#     workload_type="io",
#     max_workers=None,
#     reserve_mem_gb=50,
#     max_mem_gb=None,
#     dashboard=True
# ):

#     logical_cores = psutil.cpu_count(logical=True)
#     total_memory_gb = psutil.virtual_memory().total / 1e9

#     if max_workers is None:
#         max_workers = logical_cores

#     if max_mem_gb is None:
#         max_mem_gb = total_memory_gb

#     usable_mem_gb = max_mem_gb - reserve_mem_gb

#     if workload_type == "cpu":
#         threads_per_worker = 1
#         n_workers = min(max_workers, logical_cores)
#     elif workload_type == "io":
#         threads_per_worker = 8
#         n_workers = max(1, logical_cores // threads_per_worker)
#     else:  # "mixed"
#         threads_per_worker = 4
#         n_workers = max(1, logical_cores // threads_per_worker)

#     memory_per_worker = usable_mem_gb // n_workers

#     print(f"Number of workers = {n_workers}")
#     print(f"Number of threads per worker = {threads_per_worker}")
#     print(f"Memory per worker = {memory_per_worker}")

#     # Suppress distributed worker memory warnings
#     logging.getLogger("distributed.worker.memory").setLevel(logging.ERROR)

#     # Optionally suppress all Dask logs
#     logging.getLogger("dask").setLevel(logging.ERROR)

#         # Start client without verbose messages
#     client = Client(n_workers=n_workers,
#         threads_per_worker=threads_per_worker,
#         memory_limit=f"{int(memory_per_worker)}GB"
#     )

#     return client

# client = setup_dask_client(workload_type="cpu")
# client

In [5]:
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GULLRWF2',
            'GUNNING1',
            'BANGOWF1',
            'BANGOWF2',
            'COLWF01',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [6]:
ds = xr.open_mfdataset(boco_ds, chunks='auto', engine='h5netcdf', parallel=True)
ds = ds.chunk({'time': -1, 'lat': 50, 'lon': 50})
ds

<xarray.Dataset> Size: 32GB
Dimensions:  (time: 2832, lat: 646, lon: 1082)
Coordinates:
  * time     (time) datetime64[ns] 23kB 2015-12-19 ... 2023-12-11T23:00:00
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
    height   float64 8B ...
    crs      int32 4B ...
Data variables:
    ua100m   (time, lat, lon) float64 16GB dask.array<chunksize=(2832, 50, 50), meta=np.ndarray>
    va100m   (time, lat, lon) float64 16GB dask.array<chunksize=(2832, 50, 50), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        34f247b
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T01:43:51Z
    date_metadata_modified:    2024-10-11T01:43:51Z
    history:                   Mon Aug 05 04:51:48 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [7]:
def plot_frame(u, v, lat, lon, t, output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/',
               quiver_scale=None, extent=[147.5, 151, -38.5, -33.5], cluster=cluster, highlight_id='BOCORWF1'):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, speed, cmap='cividis', transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon, lat, u, v, scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Wind vectors at {str(t)}')
    
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    return filename


def make_filename(t, output_dir, prefix="wind"):
    try:
        # If t is datetime-like, use date formatting
        dt_str = np.datetime_as_string(t, unit='m')
        dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    except Exception:
        # If it's just an index/hour, format as hour
        if isinstance(t, (int, np.integer)):
            dt_str = f"hour{t:02d}"
        else:
            dt_str = str(t).replace(":", "").replace(" ", "_")
    
    return os.path.join(output_dir, f"{prefix}_{dt_str}.png")


In [8]:
# Crop to extent
ds_subset = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
})

ds_subset

<xarray.Dataset> Size: 783MB
Dimensions:  (time: 2832, lat: 127, lon: 136)
Coordinates:
  * time     (time) datetime64[ns] 23kB 2015-12-19 ... 2023-12-11T23:00:00
  * lon      (lon) float64 1kB 140.1 140.2 140.3 140.4 ... 154.7 154.8 154.9
  * lat      (lat) float64 1kB -41.91 -41.8 -41.69 ... -28.27 -28.16 -28.05
    height   float64 8B ...
    crs      int32 4B ...
Data variables:
    ua100m   (time, lat, lon) float64 391MB dask.array<chunksize=(2832, 4, 31), meta=np.ndarray>
    va100m   (time, lat, lon) float64 391MB dask.array<chunksize=(2832, 4, 31), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        34f247b
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T01:43:51Z
    date_metadata_modified:    2024-10-11T01:43:51Z
    history:                   Mon Aug 05 04:51:48 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [9]:
# Lazy hourly mean
hourly_composite =  ds_subset.groupby("time.hour").mean()

# Compute in parallel
with ProgressBar():
    hourly_composite = hourly_composite.compute()
    hourly_composite = hourly_composite.assign_coords(hour=("hour", np.arange(24)))

In [10]:
# # Write to NetCDF using compute=False
# delayed_obj = hourly_composite.to_netcdf(
#                 "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/hourly_uv_composite.nc",
#                 engine="h5netcdf",
#                 compute=False)

# # Trigger computation with Dask
# with ProgressBar():
#     delayed_obj.compute()

In [11]:
# extent=[140, 160, -42, -10]
# lon_min, lon_max, lat_min, lat_max = extent

# for i in range(len(hourly_composite.hour)):
#         u = hourly_composite['ua100m'].isel(hour=i)[::step, ::step]
#         v = hourly_composite['va100m'].isel(hour=i)[::step, ::step]
#         lat = hourly_composite['lat'][::step].values
#         lon = hourly_composite['lon'][::step].values
#         t = hourly_composite.hour[i].values
        
#         plot_frame(u, v, lat, lon, t, output_dir, 250, extent)

In [12]:
step = 3

lat_subset = hourly_composite['lat'].where(
    (hourly_composite['lat'] >= lat_min) & (hourly_composite['lat'] <= lat_max),
    drop=True
    ).values

lon_subset = hourly_composite['lon'].where(
    (hourly_composite['lon'] >= lon_min) & (hourly_composite['lon'] <= lon_max),
    drop=True
    ).values

for i in range(len(hourly_composite.hour)):
    u_sub = hourly_composite['ua100m'].isel(hour=i).sel(
        lat=lat_subset, lon=lon_subset
    ).values[::step, ::step]

    v_sub = hourly_composite['va100m'].isel(hour=i).sel(
        lat=lat_subset, lon=lon_subset
    ).values[::step, ::step]

    # Also subsample lat/lon to match u/v
    lat_sub = lat_subset[::step]
    lon_sub = lon_subset[::step]

    t = hourly_composite.hour[i].values

    plot_frame(u_sub, v_sub, lat_sub, lon_sub, t, output_dir, 250, extent)